# 02 — Training U-Net 2D: Blind Audio Restoration

Addestramento della U-Net 2D per restaurare mel spectrogrammi degradati da EnCodec.

**Pipeline completa:** MusicGen raw → EnCodec (artefatti RVQ) → mel spectrogram → **questo notebook** → Vocos → audio restaurato

**Strategia Blind Restoration:** il modello non riceve alcuna informazione sulla bandwidth di degradazione. Impara direttamente dai dati a stimare e rimuovere il rumore, generalizzando su tutte e 5 le bandwidth (1.5, 3.0, 6.0, 12.0, 24.0 kbps).

**Dataset:** 10.000 coppie `.pt` (2000 per bandwidth) in `train_v2/`, shape `[1, 100, 376]`.

**Flusso:**
1. Configurazione centralizzata (Cella 0)
2. Mount Drive e setup cartelle (Cella 2)
3. Dataset con random time crop (Cella 3)
4. U-Net con padding dinamico e residual learning (Cella 4)
5. Loss composita L1 + Spectral Convergence + Multi-Scale L1 (Cella 5)
6. Resume da checkpoint (Cella 6)
7. Training loop con AMP, early stopping, checkpoint (Cella 7)
8. Sanity check visivo (Cella 8)
9. Plot curve di loss (Cella 9)

## Cella 0 — Configurazione

**Unico punto di verità** per tutti gli iperparametri e i path.
Modificare qui prima di ri-eseguire qualunque altra cella.

Scelte chiave:
- `N_MELS=100`: compatibile con Vocos (`vocos-mel-24khz`), confermato in v2
- `CROP_SIZE=256`: random time crop solo in training → data augmentation + batch omogenei
- `BASE_CHANNELS=32`: 7-10M parametri, buon compromesso velocità/capacità
- `PAD_H=112`: 100 mel paddati a 112, divisibile per 2⁴=16 (4 livelli pooling)

In [1]:
# ── Path ──────────────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/audio-restoration"
DATA_DIR   = f"{DRIVE_ROOT}/data/train_v2"
CKPT_DIR   = f"{DRIVE_ROOT}/checkpoints"
PLOT_DIR   = f"{DRIVE_ROOT}/plots"

# ── Dataset ───────────────────────────────────────────────────────────────────
CROP_SIZE  = 256   # frame temporali per il random crop in training
VAL_RATIO  = 0.10  # 90/10 split
SPLIT_SEED = 42

# ── DataLoader ────────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
NUM_WORKERS  = 4

# ── Architettura ──────────────────────────────────────────────────────────────
BASE_CHANNELS = 40
PAD_H         = 112  # H=100 paddato a 112 (divisibile per 2^4=16)

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE_LR  = 5    # ReduceLROnPlateau: dimezza lr dopo N epoche senza miglioramento
LR_FACTOR    = 0.5
PATIENCE_ES  = 10   # early stopping: ferma dopo N epoche senza miglioramento
CKPT_EVERY   = 5    # salva checkpoint periodico ogni N epoche
KEEP_LAST_N  = 3    # mantieni solo gli ultimi N checkpoint periodici

# ── Loss weights ──────────────────────────────────────────────────────────────
W_L1        = 1.0   # L1 Loss standard
W_SC        = 0.5   # Spectral Convergence
W_MSL1      = 0.25  # Multi-Scale L1
MSL1_SCALES = [1, 2, 4]  # scale per avg pooling

print("Configurazione caricata.")
print(f"  DATA_DIR     : {DATA_DIR}")
print(f"  BATCH_SIZE   : {BATCH_SIZE}  EPOCHS: {EPOCHS}  LR: {LR}")
print(f"  CROP_SIZE    : {CROP_SIZE}   VAL_RATIO: {VAL_RATIO}")
print(f"  BASE_CHANNELS: {BASE_CHANNELS}  PAD_H: {PAD_H}")
print(f"  Loss weights : L1={W_L1}  SC={W_SC}  MSL1={W_MSL1}")

Configurazione caricata.
  DATA_DIR     : /content/drive/MyDrive/audio-restoration/data/train_v2
  BATCH_SIZE   : 32  EPOCHS: 50  LR: 0.001
  CROP_SIZE    : 256   VAL_RATIO: 0.1
  BASE_CHANNELS: 40  PAD_H: 112
  Loss weights : L1=1.0  SC=0.5  MSL1=0.25


## Cella 1 — Installazione dipendenze

Su Colab, PyTorch e torchaudio sono preinstallati. Installiamo solo ciò che potrebbe mancare.
La loss composita è implementata interamente con PyTorch base (no librerie esterne).

In [2]:
import subprocess, sys

# torchaudio: necessario per compatibilità con la pipeline mel
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchaudio"])

import os
import glob
import time
import random
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use("Agg")  # backend non-interattivo per Colab
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : NVIDIA A100-SXM4-40GB
Device   : cuda


## Cella 2 — Mount Google Drive e creazione cartelle

**Perché Drive è fondamentale:** Colab ha una durata massima di sessione (12h free, 24h Pro).
Salvare checkpoint su Drive garantisce che il training non vada perso in caso di disconnessione.
Ogni epoch salva `last_unet_v2.pth` → se la sessione crasha, basta ri-eseguire il notebook.

In [3]:
from google.colab import drive

# Monta Drive (force_remount=False evita rimount inutile se già montato)
drive.mount("/content/drive", force_remount=False)

# Crea le cartelle necessarie se non esistono
for d in [CKPT_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  Cartella OK: {d}")

# Conta i file disponibili nel dataset
n_pt = len(glob.glob(os.path.join(DATA_DIR, "pair_*.pt")))
print(f"\nFile .pt disponibili in {DATA_DIR}: {n_pt}")

if n_pt == 0:
    print("ATTENZIONE: nessun file trovato! Verifica che 01_data_pipeline sia stato eseguito.")
elif n_pt < 10000:
    print(f"ATTENZIONE: attesi 10000 file, trovati {n_pt}. Dataset incompleto?")
else:
    print("Dataset completo (10000 coppie).")

Mounted at /content/drive
  Cartella OK: /content/drive/MyDrive/audio-restoration/checkpoints
  Cartella OK: /content/drive/MyDrive/audio-restoration/plots

File .pt disponibili in /content/drive/MyDrive/audio-restoration/data/train_v2: 10000
Dataset completo (10000 coppie).


## Cella 3 — Dataset e DataLoader

**Random Time Crop (solo training):** il modello vede finestre di 256 frame invece di 376.
- Vantaggio 1: **data augmentation** — ogni epoch il modello vede porzioni diverse degli stessi file
- Vantaggio 2: **batch omogenei** senza collate_fn custom
- Vantaggio 3: **meno VRAM** per batch → batch size più grande

**Validation senza crop:** usa i 376 frame completi per valutazione coerente con inference.
Siccome tutti i file val hanno la stessa shape [1,100,376], il DataLoader funziona nativamente.

**Split deterministico:** `random.Random(SPLIT_SEED).shuffle()` garantisce lo stesso identico
split ad ogni resume, evitando data leakage tra sessioni.

In [4]:
class MelPairDataset(Dataset):
    """Dataset di coppie (degraded_mel, clean_mel) in formato .pt.

    Ogni file contiene un dict con:
      - 'clean_mel'    : tensor [1, 100, 376] (log-mel del segnale originale)
      - 'degraded_mel' : tensor [1, 100, 376] (log-mel dopo EnCodec)
    """

    def __init__(self, file_paths: List[str], train: bool):
        self.paths = file_paths
        self.train = train

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        try:
            data = torch.load(self.paths[idx], weights_only=True)
        except Exception as e:
            raise RuntimeError(f"Errore nel caricare {self.paths[idx]}: {e}")

        degraded = data["degraded_mel"].float()  # [1, 100, 376]
        clean    = data["clean_mel"].float()     # [1, 100, 376]

        if self.train:
            # Random Time Crop: estrae CROP_SIZE frame casuali sull'asse temporale
            T = degraded.shape[-1]  # 376
            if T > CROP_SIZE:
                start = random.randint(0, T - CROP_SIZE)
                degraded = degraded[:, :, start:start + CROP_SIZE]  # [1, 100, 256]
                clean    = clean[:, :, start:start + CROP_SIZE]
        # Validation: nessun crop, usa [1, 100, 376] completo

        return degraded, clean


# ── Split deterministico 90/10 ────────────────────────────────────────────────
all_paths = sorted(glob.glob(os.path.join(DATA_DIR, "pair_*.pt")))
assert len(all_paths) > 0, f"Nessun file .pt trovato in {DATA_DIR}"

shuffled = all_paths[:]
random.Random(SPLIT_SEED).shuffle(shuffled)  # seed fisso → stesso split ad ogni run

n_val        = max(1, int(len(shuffled) * VAL_RATIO))
n_train      = len(shuffled) - n_val
train_paths  = shuffled[:n_train]
val_paths    = shuffled[n_train:]

train_ds = MelPairDataset(train_paths, train=True)
val_ds   = MelPairDataset(val_paths,   train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,   # scarta ultimo batch incompleto → batch size sempre BATCH_SIZE
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False,  # val: mantieni tutti i campioni
)

print(f"Totale file   : {len(all_paths)}")
print(f"Train         : {n_train} campioni")
print(f"Val           : {n_val} campioni")

# Verifica shape batch
xb_train, yb_train = next(iter(train_loader))
xb_val,   yb_val   = next(iter(val_loader))
print(f"Batch train   : degraded={xb_train.shape}  clean={yb_train.shape}")
print(f"Batch val     : degraded={xb_val.shape}  clean={yb_val.shape}")
assert xb_train.shape[-1] == CROP_SIZE,     f"Train crop errato: {xb_train.shape[-1]} != {CROP_SIZE}"
assert xb_val.shape[-1]   == 376,           f"Val shape errata: {xb_val.shape[-1]} != 376"
assert xb_train.shape[1]  == 1,             "Atteso 1 canale"
assert xb_train.shape[2]  == 100,           f"Attese 100 mel bands, trovate {xb_train.shape[2]}"
print("Shape verificate.")

Totale file   : 10000
Train         : 9000 campioni
Val           : 1000 campioni
Batch train   : degraded=torch.Size([32, 1, 100, 256])  clean=torch.Size([32, 1, 100, 256])
Batch val     : degraded=torch.Size([32, 1, 100, 376])  clean=torch.Size([32, 1, 100, 376])
Shape verificate.


## Cella 4 — Architettura U-Net 2D

**Perché U-Net per i mel spectrogrammi?**
Gli artefatti di quantizzazione RVQ di EnCodec sono distribuiti su tutte le frequenze (specialmente le alte). La struttura encoder-decoder con skip connections permette di:
1. Comprimere la rappresentazione per catturare pattern globali (encoder)
2. Ricostruire i dettagli fini usando le skip connections (decoder)
3. Correggere artefatti locali ad alta frequenza

**Scelte architetturali:**
- **GroupNorm** (non BatchNorm): stabile con batch size piccoli su GPU singola; GroupNorm non dipende dalla dimensione del batch
- **GELU** (non ReLU): gradienti più morbidi, meglio per segnali audio che hanno valori negativi (log-mel)
- **Residual learning** (`output = input - noise_estimate`): converge più velocemente perché input e output sono già molto simili (LSD baseline ~0.5). Il modello impara solo la differenza
- **Padding dinamico**: accetta qualsiasi W divisibile per 16 → funziona sia con W=256 (train) che W=376 (val/inference) senza modifiche

**Verifica dimensioni H con PAD_H=112:**
```
100 → pad → 112 → /2 → 56 → /2 → 28 → /2 → 14 → /2 → 7  (bottleneck)
                                                  ×2 → 14 → ×2 → 28 → ×2 → 56 → ×2 → 112 → crop → 100
```

In [5]:
class ConvBlock(nn.Module):
    """Blocco base: Conv → GroupNorm → GELU × 2."""

    def __init__(self, in_ch: int, out_ch: int, groups: int = 8):
        super().__init__()
        # groups deve dividere out_ch — gestiamo il caso out_ch < groups
        g = min(groups, out_ch)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UNet2D(nn.Module):
    """
    U-Net 2D per restauro mel spectrogram (blind — nessun conditioning).

    Input  : [B, 1, 100, T]  dove T=256 (training) o T=376 (val/inference)
    Output : [B, 1, 100, T]  stessa shape dell'input

    Canali encoder : 1 → 32 → 64 → 128 → 256
    Bottleneck     : 256 → 512 → 256
    Canali decoder : 256 → 128 → 64 → 32 → 1 (head)
    """

    def __init__(self, base: int = BASE_CHANNELS):
        super().__init__()
        b = base  # 32

        # ── Encoder ───────────────────────────────────────────────────────────
        self.enc1 = ConvBlock(1,      b)      # 1   → 32
        self.enc2 = ConvBlock(b,      b*2)    # 32  → 64
        self.enc3 = ConvBlock(b*2,    b*4)    # 64  → 128
        self.enc4 = ConvBlock(b*4,    b*8)    # 128 → 256
        self.pool = nn.MaxPool2d(2)            # dimezza H e W

        # ── Bottleneck ────────────────────────────────────────────────────────
        self.bot = nn.Sequential(
            nn.Conv2d(b*8,  b*16, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, b*16),
            nn.GELU(),
            nn.Conv2d(b*16, b*8,  kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, b*8),
            nn.GELU(),
        )  # 256 → 512 → 256

        # ── Decoder ───────────────────────────────────────────────────────────
        # Ogni UpBlock: ConvTranspose (upsample) → cat(skip) → ConvBlock
        self.up4   = nn.ConvTranspose2d(b*8,  b*4,  kernel_size=2, stride=2)
        self.dec4  = ConvBlock(b*4 + b*8,  b*4)   # 128 + 256 → 128
        self.up3   = nn.ConvTranspose2d(b*4,  b*2,  kernel_size=2, stride=2)
        self.dec3  = ConvBlock(b*2 + b*4,  b*2)   # 64  + 128 → 64
        self.up2   = nn.ConvTranspose2d(b*2,  b,    kernel_size=2, stride=2)
        self.dec2  = ConvBlock(b   + b*2,  b)     # 32  + 64  → 32
        self.up1   = nn.ConvTranspose2d(b,    b//2, kernel_size=2, stride=2)
        self.dec1  = ConvBlock(b//2 + b,   b)     # 16  + 32  → 32

        # ── Head ──────────────────────────────────────────────────────────────
        self.head  = nn.Conv2d(b, 1, kernel_size=1)  # 32 → 1 (nessuna attivazione)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [B, 1, 100, T]
        _, _, H_orig, W_orig = x.shape

        # ── Step 1: Pad H da 100 a PAD_H=112 (divisibile per 2^4=16) ─────────
        # +6 sopra, +6 sotto → 100 + 12 = 112
        x_pad = F.pad(x, (0, 0, 6, 6))  # [B, 1, 112, T]

        # ── Step 2: Pad W se non divisibile per 16 ────────────────────────────
        # W=256 → pad_w=0 (256 = 16×16, già ok)
        # W=376 → pad_w=8 → 384 = 16×24
        _, _, H_pad, W_pad = x_pad.shape
        pad_w = (16 - W_pad % 16) % 16
        if pad_w > 0:
            x_pad = F.pad(x_pad, (0, pad_w))  # padding a destra

        # ── Step 3: Encoder ───────────────────────────────────────────────────
        s1 = self.enc1(x_pad)          # [B, 32,  112, W]
        s2 = self.enc2(self.pool(s1))  # [B, 64,   56, W/2]
        s3 = self.enc3(self.pool(s2))  # [B, 128,  28, W/4]
        s4 = self.enc4(self.pool(s3))  # [B, 256,  14, W/8]

        # ── Step 4: Bottleneck ────────────────────────────────────────────────
        bt = self.bot(self.pool(s4))   # [B, 256,   7, W/16]

        # ── Step 5: Decoder con skip connections ──────────────────────────────
        d4 = self.dec4(torch.cat([self.up4(bt), s4], dim=1))  # [B, 128, 14, W/8]
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))  # [B, 64,  28, W/4]
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))  # [B, 32,  56, W/2]
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))  # [B, 32, 112, W]

        # ── Step 6: Head → stima del rumore ───────────────────────────────────
        noise_est = self.head(d1)  # [B, 1, 112, W+pad_w]

        # ── Step 7: Rimuovi padding → ritorna a shape originale ───────────────
        noise_est = noise_est[:, :, 6:6 + H_orig, :W_orig]  # [B, 1, 100, T]

        # ── Step 8: Residual learning ─────────────────────────────────────────
        # Il modello stima il rumore da rimuovere, non direttamente il segnale pulito.
        # Converge più velocemente perché input e target sono già molto simili.
        return x - noise_est


# ── Inizializzazione e test ────────────────────────────────────────────────────
model = UNet2D().to(DEVICE)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametri totali    : {n_total:,}  ({n_total/1e6:.2f}M)")
print(f"Parametri trainabili: {n_trainable:,}")

# Test forward con entrambe le shape (train e val/inference)
for T_test, label in [(CROP_SIZE, "train (crop)"), (376, "val/inference (full)")]:
    dummy = torch.randn(2, 1, 100, T_test, device=DEVICE)
    with torch.no_grad():
        out = model(dummy)
    assert out.shape == dummy.shape, f"Shape mismatch: {out.shape} != {dummy.shape}"
    print(f"Forward {label}: input={dummy.shape} → output={out.shape}  OK")

if not (7e6 <= n_total <= 10e6):
    print(f"AVVISO: parametri fuori range target (7-10M): {n_total/1e6:.2f}M")
else:
    print(f"Parametri nel range target (7-10M).")

Parametri totali    : 7,039,101  (7.04M)
Parametri trainabili: 7,039,101
Forward train (crop): input=torch.Size([2, 1, 100, 256]) → output=torch.Size([2, 1, 100, 256])  OK
Forward val/inference (full): input=torch.Size([2, 1, 100, 376]) → output=torch.Size([2, 1, 100, 376])  OK
Parametri nel range target (7-10M).


## Cella 5 — Loss composita

**Perché una loss composita?** La L1 standard misura l'errore puntuale, ma non cattura la struttura spettrale globale. La loss composita combina tre termini complementari:

1. **L1 Loss** (peso 1.0): errore assoluto puntuale — robusto agli outlier spettrali, corregge le singole celle del mel spectrogram

2. **Spectral Convergence** (peso 0.5): norma di Frobenius normalizzata della differenza. Misura quanto la struttura globale dello spettro converge al target, indipendentemente dalla scala. Ispirata al [Music Source Separation Challenge 2025](https://arxiv.org/abs/2601.04343):
   ```
   SC = ||target - pred||_F / (||target||_F + ε)
   ```

3. **Multi-Scale L1** (peso 0.25, scale [1,2,4]): applica avg_pool2d a scale diverse prima di calcolare L1. Forza il modello a essere preciso sia sui dettagli fini (scala 1) che sulla struttura globale (scala 4). Particolarmente utile per catturare artefatti che si manifestano a scale diverse.

In [6]:
class CompositeLoss(nn.Module):
    """
    Loss composita per restauro mel spectrogram.
    Combina L1, Spectral Convergence e Multi-Scale L1.
    Tutti i componenti implementati con PyTorch base (nessuna dipendenza esterna).
    """

    def __init__(
        self,
        w_l1: float       = W_L1,
        w_sc: float       = W_SC,
        w_msl1: float     = W_MSL1,
        msl1_scales: list = MSL1_SCALES,
    ):
        super().__init__()
        self.w_l1   = w_l1
        self.w_sc   = w_sc
        self.w_msl1 = w_msl1
        self.scales = msl1_scales

    def _l1(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """L1 Loss standard: errore assoluto medio puntuale."""
        return F.l1_loss(pred, target)

    def _spectral_convergence(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Spectral Convergence: norma di Frobenius normalizzata.
        SC = ||target - pred||_F / (||target||_F + ε)
        Misura la convergenza strutturale globale dello spettro.
        """
        diff_norm   = torch.norm(target - pred, p="fro")
        target_norm = torch.norm(target,        p="fro")
        return diff_norm / (target_norm + 1e-7)

    def _multi_scale_l1(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Multi-Scale L1: media di L1 a scale diverse tramite avg_pool2d.
        Scala 1: dettagli fini; Scala 4: struttura globale.
        """
        total = torch.tensor(0.0, device=pred.device, dtype=pred.dtype)
        for s in self.scales:
            if s == 1:
                # Scala 1: nessun pooling, uguale a L1 standard
                total = total + F.l1_loss(pred, target)
            else:
                pred_ds   = F.avg_pool2d(pred,   kernel_size=s)
                target_ds = F.avg_pool2d(target, kernel_size=s)
                total = total + F.l1_loss(pred_ds, target_ds)
        return total / len(self.scales)

    def forward(
        self, pred: torch.Tensor, target: torch.Tensor
    ) -> Tuple[torch.Tensor, dict]:
        """
        Calcola la loss composita.

        Returns:
            total_loss : scalare differenziabile
            components : dict con i singoli valori (per logging)
        """
        l1   = self._l1(pred, target)
        sc   = self._spectral_convergence(pred, target)
        msl1 = self._multi_scale_l1(pred, target)

        total = self.w_l1 * l1 + self.w_sc * sc + self.w_msl1 * msl1

        return total, {"l1": l1.item(), "sc": sc.item(), "msl1": msl1.item()}


# ── Test della loss ────────────────────────────────────────────────────────────
criterion = CompositeLoss()

# Tensori dummy per verifica
dummy_pred   = torch.randn(2, 1, 100, CROP_SIZE)
dummy_target = torch.randn(2, 1, 100, CROP_SIZE)

loss_val, components = criterion(dummy_pred, dummy_target)

print(f"Loss totale : {loss_val.item():.6f}  (scalare: {loss_val.shape == torch.Size([])})")
print(f"  L1        : {components['l1']:.6f}  (peso {W_L1})")
print(f"  SC        : {components['sc']:.6f}  (peso {W_SC})")
print(f"  MSL1      : {components['msl1']:.6f}  (peso {W_MSL1})")

# Verifica differenziabilità
dummy_pred.requires_grad_(True)
loss_val2, _ = criterion(dummy_pred, dummy_target)
loss_val2.backward()
assert dummy_pred.grad is not None, "Gradiente non calcolato!"
print("Differenziabilità: OK (backward passa senza errori)")

Loss totale : 2.000380  (scalare: True)
  L1        : 1.129685  (peso 1.0)
  SC        : 1.413379  (peso 0.5)
  MSL1      : 0.656023  (peso 0.25)
Differenziabilità: OK (backward passa senza errori)


## Cella 6 — Resume da checkpoint

**Perché il resume è fondamentale su Colab:** le sessioni gratuite durano max 12h e possono crashare. Il notebook salva `last_unet_v2.pth` ad ogni epoca con TUTTO lo stato necessario per riprendere esattamente dove ci si era fermati:
- `model_state_dict`: pesi del modello
- `optimizer_state_dict`: momenti di AdamW (fondamentale — senza, AdamW riparte da zero)
- `scheduler_state_dict`: stato di ReduceLROnPlateau (contatore epoche senza miglioramento)
- `scaler_state_dict`: stato dell'AMP GradScaler
- `epoch`, `best_val_loss`, `patience_counter`: variabili di controllo del training loop

**Strategia dual-checkpoint:**
- `best_unet_v2.pth`: solo i pesi del best model → usato per inference
- `last_unet_v2.pth`: stato completo → usato per resume

In [8]:
# ── Inizializza ottimizzatore e scheduler ──────────────────────────────────────
# (devono esistere prima del resume per caricare i loro state_dict)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_FACTOR,
    patience=PATIENCE_LR,

)
scaler = torch.amp.GradScaler("cuda")

# ── Variabili di stato del training ───────────────────────────────────────────
start_epoch     = 0
best_val_loss   = float("inf")
patience_counter = 0
train_losses    = []
val_losses      = []

# ── Tentativo di resume ────────────────────────────────────────────────────────
last_ckpt_path = os.path.join(CKPT_DIR, "last_unet_v2.pth")

if os.path.exists(last_ckpt_path):
    print(f"Checkpoint trovato: {last_ckpt_path}")
    try:
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])

        start_epoch      = ckpt["epoch"]
        best_val_loss    = ckpt["best_val_loss"]
        patience_counter = ckpt["patience_counter"]
        train_losses     = ckpt.get("train_losses", [])
        val_losses       = ckpt.get("val_losses",   [])

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"  Riprendendo da epoca    : {start_epoch + 1}/{EPOCHS}")
        print(f"  Best val_loss finora    : {best_val_loss:.6f}")
        print(f"  Patience counter        : {patience_counter}/{PATIENCE_ES}")
        print(f"  Learning rate corrente  : {current_lr:.2e}")
        print(f"  Epoche già completate   : {len(train_losses)}")

    except Exception as e:
        print(f"ERRORE nel caricare il checkpoint: {e}")
        print("Training da zero.")
else:
    print("Nessun checkpoint trovato — training da zero (epoca 1).")

Nessun checkpoint trovato — training da zero (epoca 1).


## Cella 8 — Sanity Check Visivo

**Eseguire questa cella PRIMA della Cella 7.** Definisce la funzione `plot_sanity_check` che viene chiamata nel training loop.

Il sanity check visivo permette di capire immediatamente se il modello sta imparando qualcosa di sensato: se i subplot `Predicted` e `Target Clean` iniziano a somigliarsi, il training procede correttamente. Se `Predicted` appare rumoroso o identico a `Degraded`, c'è un problema.

Vengono salvati PNG in `PLOT_DIR` con nome `sanity_epoch_{e:03d}.png`.

In [9]:
def plot_sanity_check(
    model: nn.Module,
    val_loader: DataLoader,
    epoch: int,
    device: torch.device = DEVICE,
) -> None:
    """
    Genera e salva un plot 1×3: Degraded | Target Clean | Predicted.
    Chiamata ogni SANITY_EVERY epoche e alla fine del training.
    """
    model.eval()

    # Prende un singolo batch dalla validation
    with torch.no_grad():
        degraded_b, clean_b = next(iter(val_loader))
        degraded_b = degraded_b.to(device)
        with torch.amp.autocast("cuda"):
            pred_b = model(degraded_b)

    # Estrae il primo campione del batch
    deg   = degraded_b[0, 0].cpu().float().numpy()   # [100, 376]
    clean = clean_b[0, 0].cpu().float().numpy()      # [100, 376]
    pred  = pred_b[0, 0].cpu().float().numpy()       # [100, 376]

    # Scala comune per confronto visivo diretto
    vmin = min(deg.min(), clean.min(), pred.min())
    vmax = max(deg.max(), clean.max(), pred.max())

    # Metriche L1 per il titolo
    l1_deg  = float(np.abs(deg  - clean).mean())
    l1_pred = float(np.abs(pred - clean).mean())

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    panels = [
        ("Degraded (input)",           deg),
        ("Target Clean (ground truth)", clean),
        (f"Predicted — Epoch {epoch}",  pred),
    ]
    for ax, (title, panel) in zip(axes, panels):
        im = ax.imshow(
            panel, aspect="auto", origin="lower",
            vmin=vmin, vmax=vmax, cmap="magma"
        )
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Frame temporale")
        ax.set_ylabel("Banda Mel")
        plt.colorbar(im, ax=ax)

    fig.suptitle(
        f"Sanity Check — Epoch {epoch}  |  "
        f"L1(degraded, clean)={l1_deg:.4f}  →  L1(pred, clean)={l1_pred:.4f}",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()

    save_path = os.path.join(PLOT_DIR, f"sanity_epoch_{epoch:03d}.png")
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"  Sanity check → {os.path.basename(save_path)}  "
          f"(L1: {l1_deg:.4f} → {l1_pred:.4f})")

    model.train()  # ripristina modalità training


print("plot_sanity_check definita.")
print("IMPORTANTE: eseguire questa cella PRIMA della Cella 7 (training loop).")

plot_sanity_check definita.
IMPORTANTE: eseguire questa cella PRIMA della Cella 7 (training loop).


## Cella 7 — Training Loop

**AMP (Automatic Mixed Precision):** usa float16 dove possibile → ~1.5× speedup su T4, meno VRAM. Il `GradScaler` evita underflow numerici durante il backward pass in float16.

**Strategia checkpoint:**
- `last_unet_v2.pth`: sovrascritto ogni epoca → permette resume
- `best_unet_v2.pth`: aggiornato solo quando val_loss migliora → usato per inference
- `ckpt_epoch_{N:03d}.pth`: ogni `CKPT_EVERY` epoche, mantiene solo gli ultimi `KEEP_LAST_N`

**Nota:** la Cella 8 (sanity check) deve essere eseguita prima di questa.

In [10]:
SANITY_EVERY = 10  # ogni quante epoche eseguire il sanity check visivo


def save_periodic_checkpoint(epoch: int, val_loss: float) -> None:
    """Salva checkpoint periodico e rimuove i più vecchi oltre KEEP_LAST_N."""
    path = os.path.join(CKPT_DIR, f"ckpt_epoch_{epoch:03d}.pth")
    torch.save({
        "epoch":             epoch,
        "model_state_dict":  model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "best_val_loss":     best_val_loss,
        "patience_counter":  patience_counter,
        "train_losses":      train_losses,
        "val_losses":        val_losses,
    }, path)
    # Rimuovi checkpoint vecchi
    all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_epoch_*.pth")))
    for old in all_ckpts[:-KEEP_LAST_N]:
        os.remove(old)
    print(f"  Checkpoint periodico: {os.path.basename(path)}")


def save_last_checkpoint(epoch: int) -> None:
    """Salva lo stato completo per il resume. Sovrascrive ogni epoca."""
    torch.save({
        "epoch":             epoch,
        "model_state_dict":  model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "best_val_loss":     best_val_loss,
        "patience_counter":  patience_counter,
        "train_losses":      train_losses,
        "val_losses":        val_losses,
    }, os.path.join(CKPT_DIR, "last_unet_v2.pth"))


# ── Training loop ─────────────────────────────────────────────────────────────
print(f"Training da epoca {start_epoch + 1} / {EPOCHS}")
print(f"Best val_loss corrente  : {best_val_loss:.6f}")
print(f"Patience counter        : {patience_counter}/{PATIENCE_ES}")
print("-" * 75)

t0_total = time.time()
last_epoch = start_epoch  # tiene traccia dell'ultima epoca completata

for epoch in range(start_epoch + 1, EPOCHS + 1):
    t0 = time.time()

    # ── Fase Training ─────────────────────────────────────────────────────────
    model.train()
    train_loss_sum = 0.0
    for degraded, clean in train_loader:
        degraded = degraded.to(DEVICE, non_blocking=True)
        clean    = clean.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            pred       = model(degraded)
            loss, _    = criterion(pred, clean)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item()

    train_loss = train_loss_sum / len(train_loader)

    # ── Fase Validation ───────────────────────────────────────────────────────
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for degraded, clean in val_loader:
            degraded = degraded.to(DEVICE, non_blocking=True)
            clean    = clean.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                pred      = model(degraded)
                loss, _   = criterion(pred, clean)
            val_loss_sum += loss.item()

    val_loss = val_loss_sum / len(val_loader)

    # ── Scheduler step ────────────────────────────────────────────────────────
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    # ── Logging ───────────────────────────────────────────────────────────────
    elapsed = time.time() - t0
    print(
        f"Epoch {epoch:3d}/{EPOCHS}  "
        f"Train: {train_loss:.6f}  Val: {val_loss:.6f}  "
        f"LR: {current_lr:.2e}  Best: {best_val_loss:.6f}  "
        f"[{elapsed:.0f}s]"
    )

    # ── Accumula history ──────────────────────────────────────────────────────
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # ── Best model ────────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(
            {"model_state_dict": model.state_dict(), "epoch": epoch, "val_loss": val_loss},
            os.path.join(CKPT_DIR, "best_unet_v2.pth")
        )
        print(f"  Nuovo best: {best_val_loss:.6f} → best_unet_v2.pth aggiornato")
    else:
        patience_counter += 1

    # ── Checkpoint periodico ──────────────────────────────────────────────────
    if epoch % CKPT_EVERY == 0:
        save_periodic_checkpoint(epoch, val_loss)

    # ── Last checkpoint (per resume) ──────────────────────────────────────────
    save_last_checkpoint(epoch)
    last_epoch = epoch

    # ── Sanity check visivo ───────────────────────────────────────────────────
    if epoch % SANITY_EVERY == 0:
        plot_sanity_check(model, val_loader, epoch, DEVICE)

    # ── Early stopping ────────────────────────────────────────────────────────
    if patience_counter >= PATIENCE_ES:
        print(f"\nEarly stopping: nessun miglioramento per {PATIENCE_ES} epoche consecutive.")
        print(f"Training terminato all'epoca {epoch}.")
        break

# ── Fine training ─────────────────────────────────────────────────────────────
total_min = (time.time() - t0_total) / 60
print(f"\nTraining completato in {total_min:.1f} min.")
print(f"Best val_loss: {best_val_loss:.6f}")

# Sanity check finale
plot_sanity_check(model, val_loader, last_epoch, DEVICE)

Training da epoca 1 / 50
Best val_loss corrente  : inf
Patience counter        : 0/10
---------------------------------------------------------------------------
Epoch   1/50  Train: 0.528217  Val: 0.513107  LR: 1.00e-03  Best: inf  [91s]
  Nuovo best: 0.513107 → best_unet_v2.pth aggiornato
Epoch   2/50  Train: 0.514326  Val: 0.507173  LR: 1.00e-03  Best: 0.513107  [19s]
  Nuovo best: 0.507173 → best_unet_v2.pth aggiornato
Epoch   3/50  Train: 0.510612  Val: 0.506812  LR: 1.00e-03  Best: 0.507173  [22s]
  Nuovo best: 0.506812 → best_unet_v2.pth aggiornato
Epoch   4/50  Train: 0.507781  Val: 0.499418  LR: 1.00e-03  Best: 0.506812  [21s]
  Nuovo best: 0.499418 → best_unet_v2.pth aggiornato
Epoch   5/50  Train: 0.502540  Val: 0.496561  LR: 1.00e-03  Best: 0.499418  [22s]
  Nuovo best: 0.496561 → best_unet_v2.pth aggiornato
  Checkpoint periodico: ckpt_epoch_005.pth
Epoch   6/50  Train: 0.500197  Val: 0.499502  LR: 1.00e-03  Best: 0.496561  [22s]
Epoch   7/50  Train: 0.498536  Val: 0.49279

## Cella 9 — Plot Curve di Loss

Visualizza l'andamento di train loss e val loss per tutta la durata del training.

**Come leggere il plot:**
- **Gap train/val piccolo:** buona generalizzazione
- **Gap train/val grande:** overfitting — considera dropout o più dati
- **Plateau precoce:** learning rate troppo basso o modello troppo piccolo
- **Spike nella val loss:** possibile problema con i dati di validazione

La linea verticale verde indica l'epoca del best model salvato.

In [11]:
assert len(train_losses) > 0, "Nessuna storia trovata. Esegui prima la Cella 7."

n_total_params = sum(p.numel() for p in model.parameters())
best_epoch_idx = val_losses.index(min(val_losses))
best_epoch_num = best_epoch_idx + 1  # +1 perché la lista inizia da 0 ma le epoche da 1

epoche = range(1, len(train_losses) + 1)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(epoche, train_losses,
        label="Train Loss", color="steelblue",  linewidth=2)
ax.plot(epoche, val_losses,
        label="Val Loss",   color="darkorange", linewidth=2)
ax.axvline(
    best_epoch_num, color="green", linestyle="--", alpha=0.75,
    label=f"Best epoch {best_epoch_num} (val={min(val_losses):.4f})"
)

ax.set_xlabel("Epoca",     fontsize=12)
ax.set_ylabel("Loss",      fontsize=12)
ax.set_title(
    "Training Curves — Blind Audio Restoration U-Net",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()

save_path = os.path.join(PLOT_DIR, "training_curves.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()

print(f"Plot salvato: {save_path}")
print(f"Best epoch        : {best_epoch_num}/{len(train_losses)}")
print(f"Best val_loss     : {min(val_losses):.6f}")
print(f"Parametri modello : {n_total_params:,}  ({n_total_params/1e6:.2f}M)")

Plot salvato: /content/drive/MyDrive/audio-restoration/plots/training_curves.png
Best epoch        : 48/50
Best val_loss     : 0.478652
Parametri modello : 7,039,101  (7.04M)
